In [0]:
%restart_python

In [0]:
import sys
ledgr_path = "/Workspace/Users/sreelakshmitd97@gmail.com/ledgr"
if ledgr_path not in sys.path:
    sys.path.append(ledgr_path)
print(f"Added to sys.path: {ledgr_path}")

In [0]:
from ledgr_databricks.silver_transform import (
    explode_bronze_sessions, extract_call_fields,
    validate_config_coverage, compute_injection_probability,
    validate_pricing_coverage, compute_execution_cost
)
from delta.tables import DeltaTable

# Step 1: Read Bronze, explode + normalize
bronze_df = spark.table("ledgr.bronze.sessions_raw")
exploded_df = explode_bronze_sessions(bronze_df)
normalized_df = extract_call_fields(exploded_df)
print(f"Normalized (call-level) rows: {normalized_df.count()}")

# Step 2: Fail-loud validation before any injection/pricing logic runs
validate_config_coverage(normalized_df)
validate_pricing_coverage(normalized_df)
print("Config and pricing coverage validated")

# Step 3: Compute injection probability, keep only real rows for now 
# (synthetic row materialization is a separate, more involved step, see below)
injected_flagged_df = compute_injection_probability(normalized_df)

# Step 4: Compute cost
priced_df = compute_execution_cost(injected_flagged_df)

# Step 5: Write to Silver as an idempotent Delta table
silver_table = "ledgr.silver.calls_enriched"

if spark.catalog.tableExists(silver_table):
    delta_table = DeltaTable.forName(spark, silver_table)
    (delta_table.alias("target")
        .merge(priced_df.alias("source"), "target.attempt_id = source.attempt_id")
        .whenNotMatchedInsertAll()
        .execute())
    print("Merged into existing Silver table")
else:
    priced_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    print(f"Created Silver table: {silver_table}")

result_count = spark.table(silver_table).count()
print(f"Silver table row count: {result_count}")
print(f"Expected (matches Bronze explode count): 241473")

In [0]:
from pyspark.sql import functions as F
from ledgr_databricks.silver_transform import generate_synthetic_retries

# Read the real Silver data we already materialized
real_df = spark.table("ledgr.silver.calls_enriched")

# Generate synthetic failed-attempt rows for the rows flagged for injection
synthetic_df = generate_synthetic_retries(real_df)
print(f"Synthetic rows generated: {synthetic_df.count()}")

# Prepare the real rows with matching schema for the union: 
# add is_synthetic_retry=False, drop the intermediate calibration columns
final_real_df = real_df.select(
    "task_id", "run_id", "trace_id", "call_id", "attempt_id",
    "start_time", "end_time", "status_code", "status_message",
    "model_request", "model_response", "input_tokens", "output_tokens",
    "provider", "input_message_length", "output_message_length",
    "has_tool_definitions", "harness", "benchmark", "success",
    "execution_cost_usd"
).withColumn("is_synthetic_retry", F.lit(False))

# Union real + synthetic
combined_df = final_real_df.unionByName(synthetic_df)
print(f"Combined rows (real + synthetic): {combined_df.count()}")
print(f"Real rows: {final_real_df.count()}, Synthetic rows: {synthetic_df.count()}")

In [0]:
from ledgr_databricks.silver_transform import generate_synthetic_retries, compute_execution_cost
from pyspark.sql import functions as F

real_df = spark.table("ledgr.silver.calls_enriched")

synthetic_df = generate_synthetic_retries(real_df)
synthetic_df_corrected = compute_execution_cost(synthetic_df.drop("execution_cost_usd"))

final_real_df = real_df.select(
    "task_id", "run_id", "trace_id", "call_id", "attempt_id",
    "start_time", "end_time", "status_code", "status_message",
    "model_request", "model_response", "input_tokens", "output_tokens",
    "provider", "input_message_length", "output_message_length",
    "has_tool_definitions", "harness", "benchmark", "success",
    "execution_cost_usd"
).withColumn("is_synthetic_retry", F.lit(False))

combined_df_final = final_real_df.unionByName(synthetic_df_corrected)

combined_count = combined_df_final.count()
print("Combined rows: " + str(combined_count))

sample_check = combined_df_final.filter(F.col("is_synthetic_retry") == True).select(
    "call_id", "start_time", "end_time", "input_tokens", "output_tokens", "execution_cost_usd"
).limit(5)
sample_check.show(truncate=False)

In [0]:
# Verify the real timing-inversion guarantee: synthetic end_time must be 
# strictly before the ORIGINAL real row's start_time
check_df = synthetic_df_corrected.join(
    final_real_df.select("call_id", F.col("start_time").alias("real_start_time")),
    on="call_id"
)
inversions = check_df.filter(
    F.to_timestamp(F.col("end_time")) >= F.to_timestamp(F.col("real_start_time"))
).count()
print("Timing inversions found: " + str(inversions))
print("Expected: 0")

In [0]:
silver_table = "ledgr.silver.calls_enriched"

(combined_df_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table))

print("Rewrote Silver table: " + silver_table)

final_count = spark.table(silver_table).count()
print("Final row count: " + str(final_count) + ", expected 265311")

synthetic_count = spark.table(silver_table).filter(F.col("is_synthetic_retry") == True).count()
real_count = spark.table(silver_table).filter(F.col("is_synthetic_retry") == False).count()
print("Synthetic: " + str(synthetic_count) + ", Real: " + str(real_count))

In [0]:
# Idempotency test: rerun the full Silver pipeline from Bronze, confirm identical output
from ledgr_databricks.silver_transform import (
    explode_bronze_sessions, extract_call_fields, validate_config_coverage,
    compute_injection_probability, validate_pricing_coverage, compute_execution_cost,
    generate_synthetic_retries
)

bronze_df = spark.table("ledgr.bronze.sessions_raw")
exploded_df = explode_bronze_sessions(bronze_df)
normalized_df = extract_call_fields(exploded_df)
validate_config_coverage(normalized_df)
validate_pricing_coverage(normalized_df)
injected_df = compute_injection_probability(normalized_df)
priced_df = compute_execution_cost(injected_df)

synthetic_rerun = generate_synthetic_retries(priced_df)
synthetic_rerun_priced = compute_execution_cost(synthetic_rerun.drop("execution_cost_usd"))

real_rerun = priced_df.select(
    "task_id", "run_id", "trace_id", "call_id", "attempt_id",
    "start_time", "end_time", "status_code", "status_message",
    "model_request", "model_response", "input_tokens", "output_tokens",
    "provider", "input_message_length", "output_message_length",
    "has_tool_definitions", "harness", "benchmark", "success",
    "execution_cost_usd"
).withColumn("is_synthetic_retry", F.lit(False))

combined_rerun = real_rerun.unionByName(synthetic_rerun_priced)

rerun_count = combined_rerun.count()
print("Rerun row count: " + str(rerun_count))
print("Original stored row count: " + str(spark.table("ledgr.silver.calls_enriched").count()))
print("Match: " + str(rerun_count == spark.table("ledgr.silver.calls_enriched").count()))

In [0]:
from ledgr_databricks.silver_transform import add_outcome_state
from pyspark.sql import functions as F

silver_df = spark.table("ledgr.silver.calls_enriched")
silver_with_state = add_outcome_state(silver_df)

silver_with_state.groupBy("outcome_state", "is_synthetic_retry").count().show()

In [0]:
silver_table = "ledgr.silver.calls_enriched"

(silver_with_state.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table))

final_count = spark.table(silver_table).count()
print("Final row count: " + str(final_count) + ", expected 265311")


In [0]:
from ledgr_databricks.silver_transform import (
    generate_synthetic_retries, compute_execution_cost, validate_synthetic_retries
)
from pyspark.sql import functions as F

real_df = spark.table("ledgr.silver.calls_enriched").filter(F.col("is_synthetic_retry") == False)

# Note: real_df from the stored table won't have is_selected_for_injection anymore
# since that intermediate column was dropped before the final write. 
# For this validation, we instead check the ALREADY-STORED synthetic rows 
# against the already-stored real rows directly.

synthetic_stored = spark.table("ledgr.silver.calls_enriched").filter(F.col("is_synthetic_retry") == True)
real_stored = spark.table("ledgr.silver.calls_enriched").filter(F.col("is_synthetic_retry") == False)

is_valid = validate_synthetic_retries(synthetic_stored, real_stored)
print("Synthetic rows validated: " + str(is_valid))